In [1]:
%jsroot on
TCanvas*c1=new TCanvas("c1","c1",600,400);
TF1 *fcal = new TF1("fcal","[0] + [1]*x",0,4000);
fcal->SetParameters(0.2, 0.5);
fcal->SetTitle("Linear calibration;Channel;Energy (keV)");
fcal->Draw();
c1->Draw();

#### 高斯分布$f(x)=\frac{1}{\sqrt[]{2\pi}\sigma } e^{-\frac{1}{2}\frac{(x-\mu)^{2}}{\sigma ^{2}}  }$

In [3]:
TF1*fgauss = new TF1("fgauss","[0]*exp(-0.5*(pow(((x-[1])/[2]),2)))",400,700);
fgauss->SetParameters(1000,550,20);
fgauss->SetTitle("Gauss curve;Energy(kev);Counts");
fgauss->Draw();
c1->Draw();

#### TGraph 是用来画二维散点图的
#### Draw("APL");其中"A"是画坐标轴，"P"是画散点，"L"是把散点连起来

In [5]:
double ch[5] = {512, 1024, 1538, 2049, 2561};
double E[5]  = {122.1, 245.4, 367.8, 489.0, 612.3};
TGraph* gcal = new TGraph(5,ch,E);
gcal->SetTitle("Energy calibration;channel;Energy");
gcal->SetMarkerStyle(20);
gcal->SetMarkerColor(kRed);
gcal->Draw("AP");
c1->Draw();

In [6]:
TF1*flin=new TF1("flin","[0]*x+[1]");
gcal->Fit(flin);
c1->Draw();

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1.09576
NDf                       =            3
Edm                       =  2.29609e-20
NCalls                    =           40
p0                        =     0.238922   +/-   0.000373056 
p1                        =     0.143912   +/-   0.633828    


#### 用二次多项式来拟合带有高斯波动的散点图
随机数种子r(0)表示按照当前时间进行随机数生成（每次运行结果不同），r(1234)表示固定生成（每次运行结果相同）

In [8]:
TCanvas*cc =new TCanvas("cc","cc",600,400);
TF1*fquadra=new TF1("fquadra","[0]*x*x+[1]*x+[2]",0,5);
fquadra->SetParameters(1,2,3);
fquadra->Draw();
cc->Draw();

### 用向量TVector3来存储数值

In [10]:
TRandom r(0);
TVector3 v;
TGraph*g2 = new TGraph();
for(int i=0;i<5;i++)
    {
    double x = i;
    double y = fquadra->Eval(i)+r.Gaus(0,1);
    double z = fquadra->Eval(i);

    v.SetXYZ(x,y,z);
    g2->SetPoint(i,v.X(),v.Y());

    cout<<"y="<<y<<endl;
    cout<<"z="<<z<<endl;
        
    }
g2->SetMarkerStyle(21);
g2->Draw("AP");
cc->Draw();

g2->Fit(fquadra);
cc->Draw();

y=2.35856
z=3
y=5.77304
z=6
y=11.6946
z=11
y=18.3568
z=18
y=28.5822
z=27
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =     0.412336
NDf                       =            2
Edm                       =   1.0173e-19
NCalls                    =           56
p0                        =      1.02589   +/-   0.121352    
p1                        =      2.39956   +/-   0.506199    
p2                        =      2.39861   +/-   0.427324    


### 模拟一个高斯能谱

In [12]:
TH1F*hspec = new TH1F("hspec","Simualted Energy Spectrum;Energy(kev);Counts",400,0,2000);
TRandom r1(0);
TVector3 E;
for(int i=0;i<30000;i++)
    {
        double x = r1.Gaus(548,10);
        double y = r1.Gaus(1250,18);
        double z = r1.Uniform(0,2000);

        E.SetXYZ(x,y,z);

        hspec->Fill(E.X());
        hspec->Fill(E.Y());
        hspec->Fill(E.Z());
    }
hspec->Draw();
cc->Draw();

### 模拟一个反应顶点分布(二维散点)

In [14]:
TH2F * hvertex = new TH2F("hvertex","Beam spot on detector plane;x(mm);y(mm)",120,-30,30,120,-30,30);
TRandom r2(0);
TVector3 vertex;
for(int i=0;i<50000;i++)
    {
        double x = r2.Gaus(2.0, 6.0);
        double y = r2.Gaus(-1, 4);
        double z = 0;

        vertex.SetXYZ(x,y,z);

        hvertex->Fill(vertex.X(),vertex.Y());
    }
hvertex->Draw();
cc->Draw();

### 创建ROOT文件

In [16]:
TFile*fout=new TFile("root_tutorial1.root","recreate");
gcal->Write();
hspec->Write();
hvertex->Write();
fout->Close();

In [38]:
TFile *fin = new TFile("root_tutorial1.root","read");
fin->ls();

TFile**		root_tutorial1.root	
 TFile*		root_tutorial1.root	
  KEY: TGraph	Graph;1	Energy calibration
  KEY: TH1F	hspec;1	Simualted Energy Spectrum
  KEY: TH2F	hvertex;1	Beam spot on detector plane


#### 读取存储在ROOT文件的图然后画出来

In [41]:
TH1F*hread = (TH1F*)fin->Get("hspec");
hread->Draw();
c1->Draw();